In [6]:
class CampusServiceError(Exception):
    pass


class InsufficientBalanceError(CampusServiceError):
    pass

class Student:
    def __init__(self, student_id: str, name: str, balance: float = 0.0):
        self.__student_id = student_id     
        self.__name = name                 
        self.__balance = balance           

    @property
    def student_id(self) -> str:
        return self.__student_id

    @property
    def name(self) -> str:
        return self.__name
  
    @property
    def balance(self) -> float:
        return self.__balance

    def deposit(self, amount: float):
        if amount <= 0:
            raise ValueError("Deposit amount must be greater than zero.")
        self.__balance += amount
        print(f"Deposited Ksh{amount:.2f} to {self.__name}'s account. New balance: ${self.__balance:.2f}")

    def deduct(self, amount: float):
        if amount > self.__balance:
            raise InsufficientBalanceError(
                f"Error: {self.__name} has insufficient balance (Ksh{self.__balance:.2f}) for a Ksh{amount:.2f} service charge."
            )
        self.__balance -= amount

class CampusService:
    def __init__(self, service_name: str, base_fee: float):
        self.__service_name = service_name  
        self.__base_fee = base_fee          

    @property
    def service_name(self) -> str:
        return self.__service_name

    @property
    def base_fee(self) -> float:
        return self.__base_fee
 
    def process_service(self, student: Student) -> str:
        student.deduct(self.__base_fee)
        return f"Standard service '{self.__service_name}' processed for {student.name}. Fee: Ksh{self.__base_fee:.2f}"

class LibraryService(CampusService):
    def __init__(self, fine_per_day: float = 1.50):
        super().__init__(service_name="Library Book Borrowing & Fines", base_fee=0.0)
        self.__fine_per_day = fine_per_day
  
    def process_service(self, student: Student, overdue_days: int = 0) -> str:
        total_fee = overdue_days * self.__fine_per_day
        if total_fee > 0:
            student.deduct(total_fee)
            return f"[Library] Processed book return for {student.name}. Overdue fine for {overdue_days} days: Ksh{total_fee:.2f}"
        return f"[Library] Book returned on time by {student.name}. No fees incurred."

class TransportService(CampusService):
    def __init__(self, route_name: str, fare: float):
        super().__init__(service_name=f"Campus Shuttle ({route_name})", base_fee=fare)
        self.__route_name = route_name

    def process_service(self, student: Student) -> str:
        # Deduct using base fee/fare handled in parent or overridden logic
        student.deduct(self.base_fee)
        return f"[Transport] Shuttle pass issued to {student.name} for route '{self.__route_name}'. Fare: Ksh{self.base_fee:.2f}"

if __name__ == "__main__":
    print("=== Smart Campus Service System Initialization ===\n")
  
    student1 = Student(student_id="KAB001", name="Stephen Kemboi", balance=5000.00)
    print(f"Student Registered: {student1.name} (ID: {student1.student_id}) with Starting Balance: Ksh{student1.balance:.2f}\n")

    library = LibraryService(fine_per_day=200.00)
    shuttle = TransportService(route_name="Nakuru Campus - Main Gate", fare=30.00)
   
    services = [shuttle, library]
    print("--- Executing Campus Services ---")
     
    try:
        print(shuttle.process_service(student1))
    except CampusServiceError as e:
        print(e)

    print(f"Remaining Balance: Ksh{student1.balance:.2f}\n")
  
    try:
        print(library.process_service(student1, overdue_days=3))
    except CampusServiceError as e:
        print(e)

    print(f"Remaining Balance: Ksh{student1.balance:.2f}\n")

     print("--- Testing Insufficient Funds Scenario ---")
    expensive_shuttle = TransportService(route_name="Inter-City Shuttle", fare=100.00)
    
    try:
        print(expensive_shuttle.process_service(student1))
    except InsufficientBalanceError as err:
        print(f"Caught Exception: {err}")

      print("\nTop-up initiated...")
    student1.deposit(75.00)
    
    try:
        print(expensive_shuttle.process_service(student1))
    except CampusServiceError as err:
        print(err)

    print(f"\nFinal Account Balance for {student1.name}: Ksh{student1.balance:.2f}")

=== Smart Campus Service System Initialization ===

Student Registered: Stephen Kemboi (ID: KAB001) with Starting Balance: Ksh5000.00

--- Executing Campus Services ---
[Transport] Shuttle pass issued to Stephen Kemboi for route 'Nakuru Campus - Main Gate'. Fare: Ksh30.00
Remaining Balance: Ksh4970.00

[Library] Processed book return for Stephen Kemboi. Overdue fine for 3 days: Ksh600.00
Remaining Balance: Ksh4370.00

--- Testing Insufficient Funds Scenario ---
[Transport] Shuttle pass issued to Stephen Kemboi for route 'Inter-City Shuttle'. Fare: Ksh100.00

Top-up initiated...
Deposited Ksh75.00 to Stephen Kemboi's account. New balance: $4345.00
[Transport] Shuttle pass issued to Stephen Kemboi for route 'Inter-City Shuttle'. Fare: Ksh100.00

Final Account Balance for Stephen Kemboi: Ksh4245.00
